# Spectrogram-Based Song Recommendation System

This notebook uses spectrogram features extracted from Spotify previews to recommend similar songs using cosine similarity. It provides an interactive interface for selecting a song and viewing the top 40 recommendations.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, HTML
import os

# Paths
FEATURES_PATH = '/home/build/martin/spotify_v2/spectrogram_features.parquet'
METADATA_PATH = '/home/build/martin/spotify_v2/spotify_411k.parquet'

print("Loading datasets...")
df_features = pd.read_parquet(FEATURES_PATH)
df_meta = pd.read_parquet(METADATA_PATH)
print("Datasets loaded.")

Loading datasets...
Datasets loaded.


In [2]:
# Merge features with metadata
join_col = 'track_id' if 'track_id' in df_meta.columns else 'id'
df_merged = pd.merge(df_meta, df_features, left_on=join_col, right_on='track_id', how='inner')

# Identify feature columns
feature_cols = [c for c in df_merged.columns if c.startswith('spec_feat_')]
print(f"Total tracks with features: {len(df_merged)}")
print(f"Number of features: {len(feature_cols)}")

Total tracks with features: 103968
Number of features: 258


In [3]:
def get_recommendations(target_track_id, df, feature_cols, top_n=40):
    target_row = df[df['track_id'] == target_track_id]
    if target_row.empty:
        return None
    
    target_vec = target_row[feature_cols].values
    feature_matrix = df[feature_cols].values
    
    # Calculate similarity
    similarities = cosine_similarity(target_vec, feature_matrix)[0]
    
    # Get top N indices
    indices = np.argsort(similarities)[::-1]
    # Exclude target song itself
    indices = [i for i in indices if df.iloc[i]['track_id'] != target_track_id][:top_n]
    
    recs = df.iloc[indices].copy()
    recs['similarity_score'] = similarities[indices]
    
    return recs[['track_name', 'artist_name', 'similarity_score', 'track_id', 'preview_url']]

In [ ]:
# Create dropdown options (showing first 5000 for performance in dropdown)
dropdown_options = [(f"{row['track_name']} - {row['artist_name']}", row['track_id']) 
                    for idx, row in df_merged.head(5000).iterrows()]

song_dropdown = widgets.Dropdown(
    options=dropdown_options,
    description='Select Song:',
    disabled=False,
    layout={'width': '500px'}
)

output = widgets.Output()

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output()
            recs = get_recommendations(change['new'], df_merged, feature_cols)
            if recs is not None:
                display(HTML(f"<h3>Top 40 Recommendations for: {df_merged[df_merged['track_id']==change['new']]['track_name'].values[0]}</h3>"))
                display(recs)
            else:
                print("No recommendations found.")

song_dropdown.observe(on_change)

display(song_dropdown)
display(output)

Dropdown(description='Select Song:', layout=Layout(width='500px'), options=(('The Drop - Hybrid', '7KnCHIPDYw7…

Output()